# **Testing with full dataset, mean pooling, urduhack roberta, without dimensionality reduction and with articles divided into batches for capturing complete article’s semantic meaning**

In [1]:
!pip install chromadb

  Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 22.6 MB/s  0:00:01 22.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 28.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 31.6 MB/s  0:00:00 34.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 24.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.5/534.5 kB 31.5 MB/s  0:00:00
Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 32.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35/35 [chromadb]5;237m━ 34/35 [chromadb]etry-sdk]ons]


In [1]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION AND STORAGE IN CHROMADB
# Updated for 10,000 records with headline, category, and content columns
# Using MEAN POOLING and chroma_db_mean database
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time
from datetime import datetime
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in ChromaDB for efficient retrieval in recommendation systems.
    Outputs results to Word document.

    Designed for large datasets (10,000+ records) with semantic search on content.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_mean",
                 output_doc_path: str = "Urdu_News_Embeddings_Report.docx"):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path: Path to store ChromaDB locally
            output_doc_path: Path for the output Word document
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.chroma_db_path.mkdir(exist_ok=True, parents=True)
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Using device: {self.device}")
        self.add_paragraph(f"Device Information: {self.device}")

        # Load model and tokenizer with error handling
        logger.info(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name).to(self.device)
            self.model.eval()  # Set to evaluation mode
            logger.info("Model loaded successfully")
        except Exception as e:
            logger.error(f"Error loading model: {e}")
            raise

        # Initialize ChromaDB client (persistent storage)
        logger.info(f"Initializing ChromaDB at: {self.chroma_db_path}")
        self.add_paragraph(f"Initializing ChromaDB at: {self.chroma_db_path}")
        try:
            self.client = chromadb.PersistentClient(
                path=str(self.chroma_db_path)
            )
            logger.info("ChromaDB client initialized")
        except Exception as e:
            logger.error(f"Error initializing ChromaDB: {e}")
            raise

        # Create or get collection for storing embeddings
        try:
            self.collection = self.client.get_or_create_collection(
                name="urdu_news_embeddings_10k_mean",
                metadata={"hnsw:space": "cosine"}  # Use cosine similarity for recommendations
            )
            logger.info(f"Collection '{self.collection.name}' created/retrieved")
        except Exception as e:
            logger.error(f"Error creating collection: {e}")
            raise

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Embeddings Generation Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text=None, style=None):
        """Add paragraph to document. If text is None, adds empty paragraph."""
        if text is None:
            para = self.doc.add_paragraph()
        else:
            para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
            
            # Add data rows
            for row_data in data:
                cells = table.add_row().cells
                for col_idx, cell_data in enumerate(row_data):
                    cells[col_idx].text = str(cell_data)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'
            
            for row_idx, row_data in enumerate(data):
                cells = table.rows[row_idx].cells
                for col_idx, cell_data in enumerate(row_data):
                    cells[col_idx].text = str(cell_data)

        return table

    def save_document(self):
        """Save the Word document."""
        try:
            self.doc.save(self.output_doc_path)
            logger.info(f"Word document saved to: {self.output_doc_path}")
        except Exception as e:
            logger.error(f"Error saving document: {e}")
            raise

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply mean pooling to model output to get sentence embeddings.

        This takes the token embeddings and creates a single vector
        by taking the mean value across all tokens for each dimension,
        while ignoring padding tokens.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Mean pooled embeddings (batch_size, embedding_dim)
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Expand attention mask for broadcasting
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        # Sum embeddings along sequence dimension, ignoring padding tokens
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)

        # Count non-padding tokens for each sequence
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        # Calculate mean embeddings
        mean_embeddings = sum_embeddings / sum_mask

        return mean_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for a single text using mean pooling.

        For long articles (>512 tokens), this method splits the text into
        overlapping chunks, generates embeddings for each chunk, and then
        averages them to create a single embedding that represents the entire article.

        Args:
            text: Input Urdu text (content column)
            max_length: Maximum tokens per chunk (default: 512)
            chunk_overlap: Overlap between chunks to maintain context (default: 50)

        Returns:
            Embedding vector as numpy array (768,)
        """
        if not text or pd.isna(text) or str(text).strip() == "":
            logger.warning("Empty text provided for embedding generation")
            return np.zeros(768)  # Return zero vector for empty text

        text = str(text)
        
        # First, tokenize to check if text is longer than max_length
        try:
            tokens = self.tokenizer.encode(text, add_special_tokens=True)
        except Exception as e:
            logger.error(f"Error tokenizing text: {e}")
            return np.zeros(768)

        # If text fits in max_length, process normally
        if len(tokens) <= max_length:
            try:
                encoded_input = self.tokenizer(
                    text,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors='pt'
                )
                encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

                with torch.no_grad():
                    model_output = self.model(**encoded_input)

                embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
                embeddings = embeddings.cpu().detach().numpy()
                return embeddings[0]
            except Exception as e:
                logger.error(f"Error generating embedding: {e}")
                return np.zeros(768)

        # For long articles: Split into overlapping chunks
        chunk_size = max_length - 2  # Leave room for [CLS] and [SEP] tokens
        stride = chunk_size - chunk_overlap

        chunk_embeddings = []

        # Process text in chunks
        for i in range(0, len(tokens), stride):
            # Get chunk tokens
            chunk_tokens = tokens[i:i + chunk_size]

            # Stop if chunk is too small (less than 50 tokens)
            if len(chunk_tokens) < 50:
                break

            # Decode tokens back to text
            try:
                chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            except:
                continue

            # Generate embedding for this chunk
            try:
                encoded_input = self.tokenizer(
                    chunk_text,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors='pt'
                )
                encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

                with torch.no_grad():
                    model_output = self.model(**encoded_input)

                chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
                chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])
            except Exception as e:
                logger.warning(f"Error processing chunk {i}: {e}")
                continue

        if len(chunk_embeddings) == 0:
            logger.warning("No valid chunks processed, returning zero vector")
            return np.zeros(768)

        # Average all chunk embeddings to get final embedding
        final_embedding = np.mean(chunk_embeddings, axis=0)

        return final_embedding

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles in dataset and store in ChromaDB.

        Semantic search is performed on the content column, while headline
        and category are stored as metadata for display.

        Args:
            df: Dataframe containing articles with headline, category, and content
            content_column: Name of column containing article content (for embeddings)
            headline_column: Name of column containing article headline (metadata)
            category_column: Name of column containing article category (metadata)
        """
        self.add_heading("Embedding Generation Process", level=1)

        # Validate columns exist
        missing_columns = []
        for col, name in [(content_column, "content_column"), 
                         (headline_column, "headline_column"), 
                         (category_column, "category_column")]:
            if col not in df.columns:
                missing_columns.append(f"{name}: '{col}'")

        if missing_columns:
            error_msg = f"Missing columns: {', '.join(missing_columns)}"
            logger.error(error_msg)
            self.add_paragraph(f"Error: {error_msg}", style="Heading 2")
            return

        process_info = [
            f"Total Articles: {len(df)}",
            f"Content column: '{content_column}' (used for semantic search)",
            f"Headline column: '{headline_column}' (stored as metadata)",
            f"Category column: '{category_column}' (stored as metadata)",
            f"Pooling method: MEAN POOLING",
            f"Database: chroma_db_mean",
            f"Device: {self.device}"
        ]

        for info in process_info:
            self.add_paragraph(info)

        logger.info(f"\n{'='*70}")
        logger.info(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        logger.info(f"{'='*70}")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data for ChromaDB
        ids = []
        embeddings = []
        metadatas = []
        documents = []

        successful_articles = 0
        failed_articles = 0
        empty_content_count = 0

        for idx, row in df.iterrows():
            # Show progress every 10,000 articles (changed from 500)
            if (idx + 1) % 10000 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed if elapsed > 0 else 0
                eta = (total_articles - idx - 1) / articles_per_sec if articles_per_sec > 0 else 0
                progress_msg = f"Processed {idx + 1}/{total_articles} articles ({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)"
                logger.info(progress_msg)
                self.add_paragraph(progress_msg)

            # Get content for semantic search
            content_text = str(row[content_column]) if not pd.isna(row[content_column]) else ""

            # Skip empty content
            if len(content_text.strip()) == 0:
                empty_content_count += 1
                if empty_content_count <= 5:  # Log first 5 empty contents
                    logger.warning(f"Skipping article {idx} - empty content")
                continue

            try:
                # Generate embedding from content
                embedding = self.generate_embedding_for_text(content_text)

                # Prepare data for ChromaDB
                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings.append(embedding.tolist())

                # Store content preview (first 500 chars)
                documents.append(content_text[:500])

                # Store metadata (headline, category, and article index)
                headline = str(row.get(headline_column, "Unknown")) if not pd.isna(row.get(headline_column)) else "Unknown"
                category = str(row.get(category_column, "Unknown")) if not pd.isna(row.get(category_column)) else "Unknown"
                
                metadata = {
                    "article_index": idx,
                    "headline": headline,
                    "category": category,
                    "content_length": len(content_text),
                    "pooling_method": "mean_pooling"
                }
                metadatas.append(metadata)
                successful_articles += 1

            except Exception as e:
                failed_articles += 1
                error_msg = f"Error processing article {idx}: {str(e)[:100]}"
                if failed_articles <= 5:  # Log first 5 errors
                    logger.error(error_msg)
                continue

        # Log final progress
        if total_articles % 10000 != 0:
            elapsed = time.time() - start_time
            progress_msg = f"Processed {total_articles}/{total_articles} articles ({elapsed:.2f}s elapsed)"
            logger.info(progress_msg)
            self.add_paragraph(progress_msg)

        # Add all embeddings to ChromaDB in batches (ChromaDB has max batch size limit)
        self.add_heading("Storage in ChromaDB", level=2)
        logger.info(f"\n{'='*70}")
        logger.info(f"STORING {len(ids)} EMBEDDINGS IN CHROMADB...")
        logger.info(f"{'='*70}")

        summary_info = [
            f"Total articles processed: {total_articles}",
            f"Successfully embedded: {successful_articles}",
            f"Failed: {failed_articles}",
            f"Empty content skipped: {empty_content_count}",
            f"Total embeddings to store: {len(ids)}"
        ]
        
        for info in summary_info:
            self.add_paragraph(info)

        if len(ids) == 0:
            logger.error("No embeddings generated. Check your data.")
            self.add_paragraph("Error: No embeddings were generated. Please check your data.", style="Heading 2")
            return

        # ChromaDB has a max batch size limit (~5000), so we add in batches
        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        batch_info = []
        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            batch_msg = f"Storing batch {current_batch}/{total_batches} (items {batch_idx} to {batch_end})..."
            logger.info(batch_msg)
            batch_info.append([f"Batch {current_batch}", f"Items {batch_idx}-{batch_end}", "Completed"])

            try:
                self.collection.add(
                    ids=ids[batch_idx:batch_end],
                    embeddings=embeddings[batch_idx:batch_end],
                    documents=documents[batch_idx:batch_end],
                    metadatas=metadatas[batch_idx:batch_end]
                )
                logger.info(f"✓ Batch {current_batch} stored successfully")
            except Exception as e:
                error_msg = f"Error storing batch {current_batch}: {str(e)[:100]}"
                logger.error(error_msg)
                batch_info[-1][2] = f"Failed: {str(e)[:50]}"
                self.add_paragraph(f"Warning: {error_msg}")

        # Add batch information table to document
        self.add_heading("Batch Storage Details", level=3)
        self.add_table(batch_info, headers=["Batch", "Items Range", "Status"])

        logger.info("✓ All batches processed!")

        total_time = time.time() - start_time
        performance_data = [
            ["Total articles processed", total_articles],
            ["Successfully embedded", successful_articles],
            ["Failed articles", failed_articles],
            ["Empty content skipped", empty_content_count],
            ["Total time", f"{total_time:.2f} seconds ({total_time/60:.2f} minutes)"],
            ["Average time per article", f"{total_time/successful_articles:.4f} seconds" if successful_articles > 0 else "N/A"],
            ["Processing speed", f"{successful_articles/total_time:.2f} articles/second" if total_time > 0 else "N/A"]
        ]

        self.add_heading("Performance Metrics", level=3)
        self.add_table(performance_data, headers=["Metric", "Value"])

        logger.info(f"\n✓ Successfully stored {len(ids)} embeddings in ChromaDB")
        logger.info(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")

    def search_similar_articles(self, query_text: str, n_results: int = 5) -> dict:
        """
        Search for similar articles using query text.

        Searches based on content embeddings and returns results with
        headline, category, and full content.

        Args:
            query_text: Query text to find similar articles
            n_results: Number of similar articles to return

        Returns:
            Dictionary containing similar articles and their distances
        """
        if not query_text or str(query_text).strip() == "":
            logger.warning("Empty query provided")
            return {"ids": [], "distances": [], "documents": [], "metadatas": []}

        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Search in ChromaDB
        try:
            results = self.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=n_results
            )
            return results
        except Exception as e:
            logger.error(f"Error searching ChromaDB: {e}")
            return {"ids": [], "distances": [], "documents": [], "metadatas": []}

    def get_collection_stats(self) -> dict:
        """
        Get statistics about the ChromaDB collection.

        Returns:
            Dictionary with collection information
        """
        try:
            count = self.collection.count()
            return {
                "total_embeddings": count,
                "collection_name": self.collection.name,
                "pooling_method": "MEAN POOLING",
                "embedding_dimension": 768,
                "search_column": "content",
                "db_path": str(self.chroma_db_path),
                "distance_metric": "cosine"
            }
        except Exception as e:
            logger.error(f"Error getting collection stats: {e}")
            return {"error": str(e)}

    def generate_search_report(self, df: pd.DataFrame, query: str, n_results: int = 5):
        """
        Generate a comprehensive search report in the Word document.

        Args:
            df: Original dataframe
            query: Search query text
            n_results: Number of results to display
        """
        self.add_heading("Semantic Search Results", level=1)
        self.add_paragraph(f"Query: {query}")
        self.add_paragraph(f"Number of results requested: {n_results}")

        results = self.search_similar_articles(query_text=query, n_results=n_results)

        self.add_heading("Top Search Results", level=2)

        if results.get('ids') and len(results['ids']) > 0 and results['ids'][0]:
            for i, (doc_id, distance, document, metadata) in enumerate(zip(
                results['ids'][0],
                results.get('distances', [[]])[0] if results.get('distances') else [0] * len(results['ids'][0]),
                results.get('documents', [[]])[0] if results.get('documents') else [""] * len(results['ids'][0]),
                results.get('metadatas', [[]])[0] if results.get('metadatas') else [{}] * len(results['ids'][0])
            )):
                self.add_heading(f"Result #{i+1} (Similarity Score: {1 - distance:.4f})", level=3)

                result_data = [
                    ["Article ID", doc_id],
                    ["Headline", metadata.get('headline', 'N/A')],
                    ["Category", metadata.get('category', 'N/A')],
                    ["Content Length", f"{metadata.get('content_length', 'N/A')} characters"],
                    ["Pooling Method", metadata.get('pooling_method', 'mean_pooling')]
                ]

                self.add_table(result_data)

                # Get full content from dataframe
                article_idx = metadata.get('article_index', None)
                if article_idx is not None and article_idx < len(df):
                    try:
                        full_content = df.iloc[article_idx]['content']
                        if pd.isna(full_content):
                            full_content = "Content not available"
                    except:
                        full_content = "Content not available"
                    
                    self.add_paragraph("Article Preview:")
                    preview = str(full_content)[:500] + "..." if len(str(full_content)) > 500 else str(full_content)
                    self.add_paragraph(preview)
                else:
                    self.add_paragraph("Article Preview:")
                    self.add_paragraph(str(document)[:500] + "..." if len(str(document)) > 500 else str(document))

                self.add_paragraph()  # Add empty line between results - FIXED
        else:
            self.add_paragraph("No results found for the given query.")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    try:
        print("="*70)
        print("URDU NEWS RECOMMENDATION SYSTEM")
        print("USING MEAN POOLING AND chroma_db_mean DATABASE")
        print("OUTPUTTING TO WORD DOCUMENT")
        print("="*70)
        print("\nLoading balanced dataset...")

        # Load and clean dataset
        df = pd.read_csv("final_cleaned_urdu_news.csv")
        
        # Clean the data - handle NaN values in key columns
        print("Cleaning dataset...")
        df['Category'] = df['Category'].fillna('Unknown')
        df['Headline'] = df['Headline'].fillna('Unknown')
        df['content'] = df['content'].fillna('')
        
        # Remove any duplicate rows
        initial_count = len(df)
        df = df.drop_duplicates(subset=['Headline', 'content'], keep='first')
        final_count = len(df)
        duplicates_removed = initial_count - final_count
        
        print(f"\nDataset loaded and cleaned successfully!")
        print(f"Initial rows: {initial_count}")
        print(f"After cleaning: {final_count}")
        print(f"Duplicates removed: {duplicates_removed}")
        print(f"Dataset shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")

        # Initialize embedding generator with Word document output
        embedder = UrduNewsEmbeddingGenerator(
            model_name="urduhack/roberta-urdu-small",
            chroma_db_path="./chroma_db_mean",
            output_doc_path="Urdu_News_Embeddings_Report.docx"
        )

        # Add dataset information to document
        embedder.add_heading("Dataset Information", level=1)
        
        # Get unique categories, handle NaN values
        categories = df['Category'].unique().tolist()
        # Convert all items to string, handle NaN
        categories_str = [str(cat) if pd.notna(cat) else "Unknown" for cat in categories]
        
        dataset_info = [
            ["Total articles", len(df)],
            ["Dataset shape", f"{df.shape}"],
            ["Columns", ", ".join(df.columns.tolist())],
            ["Unique categories", df['Category'].nunique()],
            ["Categories", ", ".join(categories_str[:15]) + ("..." if len(categories_str) > 15 else "")]  # Show first 15
        ]
        embedder.add_table(dataset_info, headers=["Metric", "Value"])

        # Add data cleaning information
        cleaning_info = [
            ["Initial rows", initial_count],
            ["After cleaning", final_count],
            ["Duplicates removed", duplicates_removed],
            ["NaN in Category", df['Category'].isna().sum()],
            ["NaN in Headline", df['Headline'].isna().sum()],
            ["NaN in content", df['content'].isna().sum()]
        ]
        embedder.add_heading("Data Cleaning Summary", level=2)
        embedder.add_table(cleaning_info, headers=["Metric", "Value"])

        # Add sample data preview
        embedder.add_heading("Sample Data Preview", level=2)
        sample = df.iloc[0]
        sample_data = [
            ["Headline", sample['Headline'][:100] + "..."],
            ["Category", sample['Category']],
            ["Content Length", f"{len(str(sample['content']))} characters"],
            ["Content Preview", str(sample['content'])[:200] + "..." if len(str(sample['content'])) > 200 else str(sample['content'])]
        ]
        embedder.add_table(sample_data)

        # Generate embeddings and store in ChromaDB
        embedder.generate_embeddings_for_dataset(
            df=df,
            content_column="content",      # Semantic search on content
            headline_column="Headline",    # Store as metadata (capital H)
            category_column="Category"     # Store as metadata (capital C)
        )

        # Display collection statistics
        embedder.add_heading("ChromaDB Collection Statistics", level=1)
        stats = embedder.get_collection_stats()
        stats_data = [[key, value] for key, value in stats.items()]
        embedder.add_table(stats_data, headers=["Statistic", "Value"])

        # Test: Search for similar articles with custom query
        embedder.add_heading("Semantic Search Demonstration", level=1)

        # Custom query - you can modify this as needed
        query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"
        embedder.generate_search_report(df, query, n_results=3)

        # Add another search example
        query2 = "کرکٹ میچ میں پاکستان کی جیت"
        embedder.generate_search_report(df, query2, n_results=2)

        # Save the final document
        embedder.save_document()

        print("\n" + "="*70)
        print("✓ EMBEDDINGS GENERATION AND STORAGE COMPLETED SUCCESSFULLY!")
        print("✓ Using MEAN POOLING for embeddings")
        print("✓ Database stored as: chroma_db_mean")
        print("✓ Semantic search is now available on the content column")
        print("✓ Headlines and categories are stored as metadata")
        print(f"✓ Comprehensive report saved to: Urdu_News_Embeddings_Report.docx")
        print("="*70)

    except FileNotFoundError as e:
        print(f"\n❌ ERROR: Could not find the dataset file: {e}")
        print("Please make sure 'final_cleaned_urdu_news.csv' exists in the current directory.")
    except Exception as e:
        print(f"\n❌ ERROR: An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()

URDU NEWS RECOMMENDATION SYSTEM
USING MEAN POOLING AND chroma_db_mean DATABASE
OUTPUTTING TO WORD DOCUMENT

Loading balanced dataset...
Cleaning dataset...


2026-01-16 19:39:39,349 - INFO - Using device: cuda
2026-01-16 19:39:39,351 - INFO - Loading model: urduhack/roberta-urdu-small



Dataset loaded and cleaned successfully!
Initial rows: 111853
After cleaning: 111853
Duplicates removed: 0
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']


2026-01-16 19:39:43,204 - INFO - Model loaded successfully
2026-01-16 19:39:43,205 - INFO - Initializing ChromaDB at: chroma_db_mean
2026-01-16 19:39:43,214 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-01-16 19:39:43,361 - INFO - ChromaDB client initialized
2026-01-16 19:39:43,364 - INFO - Collection 'urdu_news_embeddings_10k_mean' created/retrieved
2026-01-16 19:39:43,397 - INFO - 
2026-01-16 19:39:43,397 - INFO - GENERATING EMBEDDINGS FOR 111853 ARTICLES
2026-01-16 19:39:43,398 - INFO - ======================================================================
2026-01-16 19:41:07,588 - INFO - Processed 10000/111853 articles (84.19s elapsed, ETA: 857.50s)
2026-01-16 19:42:22,553 - INFO - Processed 20000/111853 articles (159.15s elapsed, ETA: 730.94s)
2026-01-16 19:43:42,204 - INFO - Processed 30000/111853 articles (238.81s elapsed, ETA: 651.57s)
2026-01-16 19:44:56,239 - INFO - Processed 40000/111853 article


✓ EMBEDDINGS GENERATION AND STORAGE COMPLETED SUCCESSFULLY!
✓ Using MEAN POOLING for embeddings
✓ Database stored as: chroma_db_mean
✓ Semantic search is now available on the content column
✓ Headlines and categories are stored as metadata
✓ Comprehensive report saved to: Urdu_News_Embeddings_Report.docx


# **Recommender for Mean Pooling**

In [1]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM
# Uses pre-stored ChromaDB embeddings with MEAN POOLING to generate recommendations
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime

class UrduNewsRecommender:
    """
    Recommendation system for Urdu news articles using pre-stored MEAN POOLING embeddings.
    Connects to existing ChromaDB and generates recommendations based on queries.
    Outputs results to Word document.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_mean",
                 collection_name: str = "urdu_news_embeddings_10k_mean",
                 output_doc_path: str = "Urdu_News_Recommendations_Report.docx"):
        """
        Initialize the recommender with pre-stored MEAN POOLING embeddings.

        Args:
            model_name: HuggingFace model identifier (same as used for embedding generation)
            chroma_db_path: Path to existing ChromaDB with mean pooling embeddings
            collection_name: Name of the collection with mean pooling embeddings
            output_doc_path: Path for the output Word document
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer (for query embeddings)
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Connect to existing ChromaDB with MEAN POOLING embeddings
        print(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.add_paragraph(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(path=str(self.chroma_db_path))

        # Get existing collection with MEAN POOLING embeddings
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Connected to collection: {collection_name}")
            print(f"✓ Total articles in database: {self.collection.count()}")
            self.add_paragraph(f"✓ Connected to collection: {collection_name}")
            self.add_paragraph(f"✓ Total articles in database: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name}'")
            print(f"Make sure embeddings are generated first!")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'

        for row_idx, row_data in enumerate(data):
            if headers and row_idx == 0:
                continue  # Skip first row if headers were added
            if not headers:
                cells = table.rows[row_idx].cells
            else:
                cells = table.add_row().cells
            for col_idx, cell_data in enumerate(row_data):
                cells[col_idx].text = str(cell_data)

        return table

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply MEAN POOLING to get sentence embeddings.

        This takes the token embeddings and creates a single vector
        by taking the mean value across all tokens for each dimension,
        while ignoring padding tokens.
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Expand attention mask for broadcasting
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        # Sum embeddings along sequence dimension, ignoring padding tokens
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)

        # Count non-padding tokens for each sequence
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        # Calculate mean embeddings
        mean_embeddings = sum_embeddings / sum_mask

        return mean_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for query text using MEAN POOLING.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks

        Returns:
            Query embedding vector using MEAN POOLING
        """
        # Tokenize to check length
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # If query fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long queries: use chunking (same as article processing)
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def get_recommendations(self, query: str, n_results: int = 5,
                          filter_category: str = None) -> dict:
        """
        Get article recommendations based on Urdu query using MEAN POOLING embeddings.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            filter_category: Optional category filter (e.g., "Sports", "Business")

        Returns:
            Dictionary containing recommended articles with metadata
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS USING MEAN POOLING")
        print(f"{'='*70}")
        print(f"Query: {query}")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Generate query embedding using MEAN POOLING
        print("Generating query embedding using MEAN POOLING...")
        query_embedding = self.generate_query_embedding(query)
        print("✓ Query embedding generated")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Search in ChromaDB with MEAN POOLING embeddings
        print(f"Searching for similar articles...")
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"✓ Found {len(results['ids'][0]) if results['ids'] else 0} recommendations\n")

        return results

    def display_recommendations(self, results: dict, df: pd.DataFrame = None,
                               show_full_content: bool = False):
        """
        Display recommendations in a formatted way.

        Args:
            results: Results from get_recommendations()
            df: Optional dataframe to fetch full content
            show_full_content: Whether to display full article content
        """
        if not results['ids'] or len(results['ids'][0]) == 0:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        print(f"{'='*70}")
        print(f"TOP {len(results['ids'][0])} RECOMMENDATIONS")
        print(f"{'='*70}\n")

        self.add_heading(f"Top {len(results['ids'][0])} Recommendations", level=2)

        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            # Calculate similarity score (1 - distance for cosine)
            similarity_score = 1 - distance

            print(f"{'='*70}")
            print(f"RECOMMENDATION #{i+1}")
            print(f"{'='*70}")
            print(f"📰 Article ID: {doc_id}")
            print(f"🎯 Similarity Score: {similarity_score:.4f} ({similarity_score*100:.2f}%)")
            print(f"\n📌 HEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"📂 CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"📏 Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"🔧 Pooling Method: {metadata.get('pooling_method', 'mean_pooling')}")

            # Add to Word document
            self.add_heading(f"Recommendation #{i+1} (Similarity: {similarity_score*100:.2f}%)", level=3)

            recommendation_data = [
                ["Article ID", doc_id],
                ["Similarity Score", f"{similarity_score:.4f} ({similarity_score*100:.2f}%)"],
                ["Headline", metadata.get('headline', 'N/A')],
                ["Category", metadata.get('category', 'N/A')],
                ["Content Length", f"{metadata.get('content_length', 'N/A')} characters"],
                ["Pooling Method", metadata.get('pooling_method', 'mean_pooling')]
            ]

            self.add_table(recommendation_data)

            # Get and display article content
            article_idx = metadata.get('article_index', None)

            if df is not None and article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                if show_full_content:
                    print(f"\n📄 FULL CONTENT:")
                    print(f"{'-'*70}")
                    print(full_content)
                    self.add_paragraph("Full Content:")
                    self.add_paragraph(full_content)
                else:
                    # Display preview (first 400 characters)
                    preview = full_content[:400] + "..." if len(full_content) > 400 else full_content
                    print(f"\n📄 CONTENT PREVIEW:")
                    print(f"{'-'*70}")
                    print(preview)
                    self.add_paragraph("Content Preview:")
                    self.add_paragraph(preview)
            else:
                # Fallback to stored document preview
                print(f"\n📄 CONTENT PREVIEW:")
                print(f"{'-'*70}")
                print(document)
                self.add_paragraph("Content Preview:")
                self.add_paragraph(document)

            print(f"\n{'='*70}\n")
            self.add_paragraph('')  # Add empty line between recommendations

    def get_statistics(self) -> dict:
        """Get statistics about the recommendation system."""
        return {
            "total_articles": self.collection.count(),
            "collection_name": self.collection.name,
            "model": self.model_name,
            "device": str(self.device),
            "embedding_dimension": 768,
            "pooling_method": "MEAN POOLING"
        }

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()
        stats_data = [[key, value] for key, value in stats.items()]
        self.add_table(stats_data, headers=["Statistic", "Value"])


# =============================================================================
# MAIN EXECUTION - RECOMMENDATION SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM - MEAN POOLING EMBEDDINGS")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)

    # Load the dataset (optional - only needed for displaying full content)
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    
    # Clean the data - handle NaN values in key columns
    print("Cleaning dataset...")
    df['Category'] = df['Category'].fillna('Unknown')
    df['Headline'] = df['Headline'].fillna('Unknown')
    df['content'] = df['content'].fillna('')
    
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender (connects to existing ChromaDB with MEAN POOLING embeddings)
    print("\n" + "="*70)
    print("INITIALIZING RECOMMENDATION SYSTEM WITH MEAN POOLING")
    print("="*70)

    recommender = UrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_mean",
        collection_name="urdu_news_embeddings_10k_mean",
        output_doc_path="Urdu_News_Recommendations_Report.docx"
    )

    # Add dataset information to document with NaN handling
    recommender.add_heading("Dataset Information", level=1)
    
    # Get unique categories, handle NaN values
    categories = df['Category'].unique().tolist()
    # Convert all items to string, handle NaN
    categories_str = [str(cat) if pd.notna(cat) else "Unknown" for cat in categories]
    
    dataset_info = [
        ["Total articles", len(df)],
        ["Dataset shape", f"{df.shape}"],
        ["Unique categories", df['Category'].nunique()],
        ["Categories", ", ".join(categories_str[:15]) + ("..." if len(categories_str) > 15 else "")]  # Show first 15
    ]
    recommender.add_table(dataset_info, headers=["Metric", "Value"])

    # Display system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    for key, value in stats.items():
        print(f"{key}: {value}")

    recommender.add_statistics_to_doc()


    # =================================================================
    # EXAMPLE 3: Multiple queries (batch recommendations)
    # =================================================================
    print("\n\n" + "="*70)
    print("MULTIPLE QUERY RECOMMENDATIONS")
    print("="*70)

    recommender.add_heading("Example 3: Multiple Query Recommendations", level=1)

    queries = [
    "پاکستان میں شمسی توانائی کے منصوبوں کو فروغ دینے کے لیے حکومت نے نئی پالیسی متعارف کرائی ہے۔ اس پالیسی کے تحت گھریلو صارفین کو سستی قرضوں کی سہولت فراہم کی جائے گی۔",
    "کرکٹ ورلڈ کپ میں پاکستان کی ٹیم نے شاندار کارکردگی کا مظاہرہ کیا۔ بابر اعظم کی کپتانی میں ٹیم نے مسلسل پانچ میچ جیت کر سیمی فائنل میں جگہ بنا لی۔ تمام کھلاڑیوں نے بہترین فارم دکھائی۔",
    "لاہور میں ہونے والی بارش نے شہر کے کئی علاقوں میں سیلاب کی صورتحال پیدا کر دی۔ نکاسی آب کا نظام ناکام ہونے سے شہریوں کو شدید مشکلات کا سامنا ہے۔ انتظامیہ نے ہنگامی اقدامات شروع کر دیے۔",
    "موسمیاتی تبدیلی کے اثرات پاکستان میں تیزی سے بڑھ رہے ہیں۔ گلیشیئرز پگھلنے سے سیلاب کا خطرہ بڑھ گیا ہے۔ حکومت کو فوری طور پر موثر اقدامات کرنے کی ضرورت ہے تاکہ مستقبل محفوظ بنایا جا سکے۔",
    "ڈیجیٹل پاکستان کے منصوبے کے تحت دور دراز علاقوں میں انٹرنیٹ کی سہولیات فراہم کی جا رہی ہیں۔ اس سے تعلیم اور کاروبار کے شعبوں میں انقلاب آنے کی توقع ہے۔ نوجوانوں کو آن لائن ملازمتوں کے مواقع ملیں گے۔",
    "معیشت اور کاروبار کی خبریں",
    "کرکٹ کی تازہ ترین خبریں",
    "فلموں اور ڈرامے کی خبریں",
    "پاکستان کا انتخابی نظام",
    "پاکستانی شوبز انڈسٹری",
    "صحت اور تندرستی کے حوالے سے مفید معلومات",
    "پاکستان میں تعلیمی نظام اور جدید تربیت",
    "ٹیکنالوجی",
    "کاروبار اور معاشی ترقی کی خبریں",
    "پاکستانی سیاست اور حکومتی پالیسیاں",
    "کھیلوں اور تفریحی پروگراموں کی خبریں",
    "مذہبی تعلیمات اور روحانی معلومات",
    "پاکستان کے خوبصورت سیاحتی مقامات"
    ]

    for idx, query in enumerate(queries, 1):
        print(f"\n{'─'*70}")
        print(f"QUERY {idx}: {query}")
        print(f"{'─'*70}")

        recommender.add_heading(f"Query {idx}: {query}", level=2)

        results = recommender.get_recommendations(
            query=query,
            n_results=10  # Get top 10 for each query
        )

        # Display only headlines for compact view
        if results['ids'] and len(results['ids'][0]) > 0:
            headlines_data = []
            for i, (doc_id, distance, metadata) in enumerate(zip(
                results['ids'][0],
                results['distances'][0],
                results['metadatas'][0]
            ), 1):
                similarity = (1 - distance) * 100
                print(f"{i}. [{similarity:.1f}%] {metadata.get('headline', 'N/A')}")
                headlines_data.append([f"{i}", f"{similarity:.1f}%", metadata.get('headline', 'N/A')])

            # Add headlines table to document
            recommender.add_table(headlines_data, headers=["Rank", "Similarity", "Headline"])
        print()

    # Save the final document
    recommender.save_document()

    print("="*70)
    print("✓ RECOMMENDATION SYSTEM DEMO COMPLETED!")
    print("✓ Using MEAN POOLING embeddings for recommendations")
    print("✓ Connected to chroma_db_mean database")
    print(f"✓ Comprehensive report saved to: Urdu_News_Recommendations_Report.docx")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM - MEAN POOLING EMBEDDINGS
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
Cleaning dataset...
✓ Dataset loaded: 111853 articles

INITIALIZING RECOMMENDATION SYSTEM WITH MEAN POOLING
Using device: cuda
Loading model: urduhack/roberta-urdu-small


/home/cvl/anaconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Connecting to ChromaDB at: chroma_db_mean
✓ Connected to collection: urdu_news_embeddings_10k_mean
✓ Total articles in database: 111853

SYSTEM STATISTICS
total_articles: 111853
collection_name: urdu_news_embeddings_10k_mean
model: urduhack/roberta-urdu-small
device: cuda
embedding_dimension: 768
pooling_method: MEAN POOLING


MULTIPLE QUERY RECOMMENDATIONS

──────────────────────────────────────────────────────────────────────
QUERY 1: پاکستان میں شمسی توانائی کے منصوبوں کو فروغ دینے کے لیے حکومت نے نئی پالیسی متعارف کرائی ہے۔ اس پالیسی کے تحت گھریلو صارفین کو سستی قرضوں کی سہولت فراہم کی جائے گی۔
──────────────────────────────────────────────────────────────────────

GENERATING RECOMMENDATIONS USING MEAN POOLING
Query: پاکستان میں شمسی توانائی کے منصوبوں کو فروغ دینے کے لیے حکومت نے نئی پالیسی متعارف کرائی ہے۔ اس پالیسی کے تحت گھریلو صارفین کو سستی قرضوں کی سہولت فراہم کی جائے گی۔
Number of results: 10

Generating query embedding using MEAN POOLING...
✓ Query embedding generated
Sear

# **Testing with 10,000 dataset, MAX pooling, urduhack roberta, without dimensionality reduction and with articles divided into batches for capturing complete article’s semantic meaning**

In [6]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION AND STORAGE IN CHROMADB
# Updated for 10,000 records with headline, category, and content columns
# Using MAX POOLING and chroma_db_max database
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in ChromaDB for efficient retrieval in recommendation systems.

    Designed for large datasets (10,000+ records) with semantic search on content.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_max"):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path: Path to store ChromaDB locally
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.chroma_db_path.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB client (persistent storage)
        print(f"Initializing ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(
            path=str(self.chroma_db_path)
        )

        # Create or get collection for storing embeddings
        self.collection = self.client.get_or_create_collection(
            name="urdu_news_embeddings_10k_max",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity for recommendations
        )

    def max_pooling(self, model_output, attention_mask):
        """
        Apply max pooling to model output to get sentence embeddings.

        This takes the token embeddings and creates a single vector
        by taking the maximum value across all tokens for each dimension.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Pooled embeddings (batch_size, embedding_dim)
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Expand attention mask for broadcasting
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        # Set padding tokens to very small value so they don't affect max pooling
        token_embeddings[input_mask_expanded == 0] = -1e9

        # Apply max pooling across the sequence dimension
        max_embeddings = torch.max(token_embeddings, 1)[0]

        return max_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for a single text using max pooling.

        For long articles (>512 tokens), this method splits the text into
        overlapping chunks, generates embeddings for each chunk, and then
        averages them to create a single embedding that represents the entire article.

        Args:
            text: Input Urdu text (content column)
            max_length: Maximum tokens per chunk (default: 512)
            chunk_overlap: Overlap between chunks to maintain context (default: 50)

        Returns:
            Embedding vector as numpy array (768,)
        """
        # First, tokenize to check if text is longer than max_length
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        # If text fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.max_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long articles: Split into overlapping chunks
        chunk_size = max_length - 2  # Leave room for [CLS] and [SEP] tokens
        stride = chunk_size - chunk_overlap

        chunk_embeddings = []

        # Process text in chunks
        for i in range(0, len(tokens), stride):
            # Get chunk tokens
            chunk_tokens = tokens[i:i + chunk_size]

            # Stop if chunk is too small (less than 50 tokens)
            if len(chunk_tokens) < 50:
                break

            # Decode tokens back to text
            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)

            # Generate embedding for this chunk
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.max_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        # Average all chunk embeddings to get final embedding
        final_embedding = np.mean(chunk_embeddings, axis=0)

        return final_embedding

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles in dataset and store in ChromaDB.

        Semantic search is performed on the content column, while headline
        and category are stored as metadata for display.

        Args:
            df: Dataframe containing articles with headline, category, and content
            content_column: Name of column containing article content (for embeddings)
            headline_column: Name of column containing article headline (metadata)
            category_column: Name of column containing article category (metadata)
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Content column: '{content_column}' (used for semantic search)")
        print(f"Headline column: '{headline_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (stored as metadata)")
        print(f"Pooling method: MAX POOLING")
        print(f"Database: chroma_db_max")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data for ChromaDB
        ids = []
        embeddings = []
        metadatas = []
        documents = []

        for idx, row in df.iterrows():
            # Show progress every 500 articles (more frequent for 10k dataset)
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            # Get content for semantic search
            content_text = str(row[content_column])

            # Skip empty content
            if len(content_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty content")
                continue

            try:
                # Generate embedding from content
                embedding = self.generate_embedding_for_text(content_text)

                # Prepare data for ChromaDB
                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings.append(embedding.tolist())

                # Store content preview (first 500 chars)
                documents.append(content_text[:500])

                # Store metadata (headline, category, and article index)
                metadata = {
                    "article_index": idx,
                    "headline": str(row.get(headline_column, "Unknown")),
                    "category": str(row.get(category_column, "Unknown")),
                    "content_length": len(content_text),
                    "pooling_method": "max_pooling"
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Add all embeddings to ChromaDB in batches (ChromaDB has max batch size limit)
        print(f"\n{'='*70}")
        print(f"STORING {len(ids)} EMBEDDINGS IN CHROMADB...")
        print(f"{'='*70}")

        # ChromaDB has a max batch size limit (~5000), so we add in batches
        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            self.collection.add(
                ids=ids[batch_idx:batch_end],
                embeddings=embeddings[batch_idx:batch_end],
                documents=documents[batch_idx:batch_end],
                metadatas=metadatas[batch_idx:batch_end]
            )

        print(f"✓ All batches stored successfully!")

        total_time = time.time() - start_time
        print(f"\n✓ Successfully stored {len(ids)} embeddings in ChromaDB")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5) -> dict:
        """
        Search for similar articles using query text.

        Searches based on content embeddings and returns results with
        headline, category, and full content.

        Args:
            query_text: Query text to find similar articles
            n_results: Number of similar articles to return

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Search in ChromaDB
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about the ChromaDB collection.

        Returns:
            Dictionary with collection information
        """
        count = self.collection.count()
        return {
            "total_embeddings": count,
            "collection_name": self.collection.name,
            "pooling_method": "MAX POOLING",
            "embedding_dimension": 768,
            "search_column": "content",
            "db_path": str(self.chroma_db_path)
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Load your preprocessed dataset
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM ")
    print("USING MAX POOLING AND chroma_db_max DATABASE")
    print("="*70)
    print("\nLoading balanced dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline'][:100]}...")
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_max"
    )

    # Generate embeddings and store in ChromaDB
    embedder.generate_embeddings_for_dataset(
        df=df,
        content_column="content",      # Semantic search on content
        headline_column="Headline",    # Store as metadata (capital H)
        category_column="Category"     # Store as metadata (capital C)
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # Test: Search for similar articles with custom query
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH ON CONTENT")
    print("="*70)

    # Custom query - you can modify this as needed
    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"
    print(f"\nQuery: {query}\n")

    results = embedder.search_similar_articles(
        query_text=query,
        n_results=5
    )

    print("Top 5 most relevant articles found:")
    print("="*70)

    if results['ids'] and len(results['ids']) > 0:
        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            print(f"\n{'='*70}")
            print(f"RESULT #{i+1}")
            print(f"{'='*70}")
            print(f"Article ID: {doc_id}")
            print(f"Similarity Score: {1 - distance:.4f} (higher is better)")
            print(f"\nHEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"Pooling Method: {metadata.get('pooling_method', 'max_pooling')}")

            # Get full content from dataframe
            article_idx = metadata.get('article_index', None)
            if article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                # Display article preview (first 300 characters)
                print(f"\n--- ARTICLE PREVIEW (First 300 chars) ---")
                print(full_content[:300] + "..." if len(full_content) > 300 else full_content)

                # Optionally display full content (commented out by default to avoid clutter)
                # Uncomment the lines below if you want to see full content
                # print(f"\n--- FULL CONTENT ---")
                # print(full_content)
            else:
                print(f"\n--- ARTICLE PREVIEW ---")
                print(document)

            print(f"\n{'='*70}")
    else:
        print("No results found!")

    print("\n" + "="*70)
    print("✓ EMBEDDINGS GENERATION AND STORAGE COMPLETED SUCCESSFULLY!")
    print("✓ Using MAX POOLING for embeddings")
    print("✓ Database stored as: chroma_db_max")
    print("✓ Semantic search is now available on the content column")
    print("✓ Headlines and categories are stored as metadata")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM 
USING MAX POOLING AND chroma_db_max DATABASE

Loading balanced dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا...
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing ChromaDB at: chroma_db_max

GENERATING EMBEDDINGS FOR 111853 ARTICLES
Content column: 'content' (used for semantic search)
Headline column: 

# **Recommender System for MAX Pooling**


In [2]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM
# Uses pre-stored ChromaDB embeddings with MAX POOLING to generate recommendations
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime

class UrduNewsRecommender:
    """
    Recommendation system for Urdu news articles using pre-stored MAX POOLING embeddings.
    Connects to existing ChromaDB and generates recommendations based on queries.
    Outputs results to Word document.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_max",
                 collection_name: str = "urdu_news_embeddings_10k_max",
                 output_doc_path: str = "Urdu_News_Recommendations_Max_Pooling_Report.docx"):
        """
        Initialize the recommender with pre-stored MAX POOLING embeddings.

        Args:
            model_name: HuggingFace model identifier (same as used for embedding generation)
            chroma_db_path: Path to existing ChromaDB with max pooling embeddings
            collection_name: Name of the collection with max pooling embeddings
            output_doc_path: Path for the output Word document
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer (for query embeddings)
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Connect to existing ChromaDB with MAX POOLING embeddings
        print(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.add_paragraph(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(path=str(self.chroma_db_path))

        # Get existing collection with MAX POOLING embeddings
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Connected to collection: {collection_name}")
            print(f"✓ Total articles in database: {self.collection.count()}")
            self.add_paragraph(f"✓ Connected to collection: {collection_name}")
            self.add_paragraph(f"✓ Total articles in database: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name}'")
            print(f"Make sure embeddings are generated first!")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System Report (Max Pooling)', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'

        for row_idx, row_data in enumerate(data):
            if headers and row_idx == 0:
                continue  # Skip first row if headers were added
            if not headers:
                cells = table.rows[row_idx].cells
            else:
                cells = table.add_row().cells
            for col_idx, cell_data in enumerate(row_data):
                cells[col_idx].text = str(cell_data)

        return table

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")

    def max_pooling(self, model_output, attention_mask):
        """Apply MAX POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        token_embeddings[input_mask_expanded == 0] = -1e9
        max_embeddings = torch.max(token_embeddings, 1)[0]
        return max_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for query text using MAX POOLING.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks

        Returns:
            Query embedding vector using MAX POOLING
        """
        # Tokenize to check length
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # If query fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.max_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long queries: use chunking (same as article processing)
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.max_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def get_recommendations(self, query: str, n_results: int = 5,
                          filter_category: str = None) -> dict:
        """
        Get article recommendations based on Urdu query using MAX POOLING embeddings.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            filter_category: Optional category filter (e.g., "Sports", "Business")

        Returns:
            Dictionary containing recommended articles with metadata
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS USING MAX POOLING")
        print(f"{'='*70}")
        print(f"Query: {query}")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Generate query embedding using MAX POOLING
        print("Generating query embedding using MAX POOLING...")
        query_embedding = self.generate_query_embedding(query)
        print("✓ Query embedding generated")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Search in ChromaDB with MAX POOLING embeddings
        print(f"Searching for similar articles...")
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"✓ Found {len(results['ids'][0]) if results['ids'] else 0} recommendations\n")

        return results

    def display_recommendations(self, results: dict, df: pd.DataFrame = None,
                               show_full_content: bool = False):
        """
        Display recommendations in a formatted way.

        Args:
            results: Results from get_recommendations()
            df: Optional dataframe to fetch full content
            show_full_content: Whether to display full article content
        """
        if not results['ids'] or len(results['ids'][0]) == 0:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        print(f"{'='*70}")
        print(f"TOP {len(results['ids'][0])} RECOMMENDATIONS")
        print(f"{'='*70}\n")

        self.add_heading(f"Top {len(results['ids'][0])} Recommendations", level=2)

        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            # Calculate similarity score (1 - distance for cosine)
            similarity_score = 1 - distance

            print(f"{'='*70}")
            print(f"RECOMMENDATION #{i+1}")
            print(f"{'='*70}")
            print(f"📰 Article ID: {doc_id}")
            print(f"🎯 Similarity Score: {similarity_score:.4f} ({similarity_score*100:.2f}%)")
            print(f"\n📌 HEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"📂 CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"📏 Content Length: {metadata.get('content_length', 'N/A')} characters")

            # Add to Word document
            self.add_heading(f"Recommendation #{i+1} (Similarity: {similarity_score*100:.2f}%)", level=3)

            recommendation_data = [
                ["Article ID", doc_id],
                ["Similarity Score", f"{similarity_score:.4f} ({similarity_score*100:.2f}%)"],
                ["Headline", metadata.get('headline', 'N/A')],
                ["Category", metadata.get('category', 'N/A')],
                ["Content Length", f"{metadata.get('content_length', 'N/A')} characters"],
                ["Pooling Method", metadata.get('pooling_method', 'max_pooling')]
            ]

            self.add_table(recommendation_data)

            # Get and display article content
            article_idx = metadata.get('article_index', None)

            if df is not None and article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                if show_full_content:
                    print(f"\n📄 FULL CONTENT:")
                    print(f"{'-'*70}")
                    print(full_content)
                    self.add_paragraph("Full Content:")
                    self.add_paragraph(full_content)
                else:
                    # Display preview (first 400 characters)
                    preview = full_content[:400] + "..." if len(full_content) > 400 else full_content
                    print(f"\n📄 CONTENT PREVIEW:")
                    print(f"{'-'*70}")
                    print(preview)
                    self.add_paragraph("Content Preview:")
                    self.add_paragraph(preview)
            else:
                # Fallback to stored document preview
                print(f"\n📄 CONTENT PREVIEW:")
                print(f"{'-'*70}")
                print(document)
                self.add_paragraph("Content Preview:")
                self.add_paragraph(document)

            print(f"\n{'='*70}\n")
            self.add_paragraph('')  # Add empty line between recommendations

    def get_statistics(self) -> dict:
        """Get statistics about the recommendation system."""
        return {
            "total_articles": self.collection.count(),
            "collection_name": self.collection.name,
            "model": self.model_name,
            "device": str(self.device),
            "embedding_dimension": 768,
            "pooling_method": "MAX POOLING"
        }

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()
        stats_data = [[key, value] for key, value in stats.items()]
        self.add_table(stats_data, headers=["Statistic", "Value"])


# =============================================================================
# MAIN EXECUTION - RECOMMENDATION SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM - MAX POOLING EMBEDDINGS")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)

    # Load the dataset (optional - only needed for displaying full content)
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    
    # Clean the data - handle NaN values in key columns
    print("Cleaning dataset...")
    df['Category'] = df['Category'].fillna('Unknown')
    df['Headline'] = df['Headline'].fillna('Unknown')
    df['content'] = df['content'].fillna('')
    
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender (connects to existing ChromaDB with MAX POOLING embeddings)
    print("\n" + "="*70)
    print("INITIALIZING RECOMMENDATION SYSTEM WITH MAX POOLING")
    print("="*70)

    recommender = UrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_max",
        collection_name="urdu_news_embeddings_10k_max",
        output_doc_path="Urdu_News_Recommendations_Max_Pooling_Report.docx"
    )

    # Add dataset information to document with NaN handling
    recommender.add_heading("Dataset Information", level=1)
    
    # Get unique categories, handle NaN values
    categories = df['Category'].unique().tolist()
    # Convert all items to string, handle NaN
    categories_str = [str(cat) if pd.notna(cat) else "Unknown" for cat in categories]
    
    dataset_info = [
        ["Total articles", len(df)],
        ["Dataset shape", f"{df.shape}"],
        ["Unique categories", df['Category'].nunique()],
        ["Categories", ", ".join(categories_str[:15]) + ("..." if len(categories_str) > 15 else "")]
    ]
    recommender.add_table(dataset_info, headers=["Metric", "Value"])

    # Display system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    for key, value in stats.items():
        print(f"{key}: {value}")

    recommender.add_statistics_to_doc()

    # =================================================================
    # EXAMPLE 3: Multiple queries (batch recommendations)
    # =================================================================
    print("\n\n" + "="*70)
    print("MULTIPLE QUERY RECOMMENDATIONS")
    print("="*70)

    recommender.add_heading("Example 3: Multiple Query Recommendations", level=1)

    queries = [
    "پاکستان میں شمسی توانائی کے منصوبوں کو فروغ دینے کے لیے حکومت نے نئی پالیسی متعارف کرائی ہے۔ اس پالیسی کے تحت گھریلو صارفین کو سستی قرضوں کی سہولت فراہم کی جائے گی۔",
    "کرکٹ ورلڈ کپ میں پاکستان کی ٹیم نے شاندار کارکردگی کا مظاہرہ کیا۔ بابر اعظم کی کپتانی میں ٹیم نے مسلسل پانچ میچ جیت کر سیمی فائنل میں جگہ بنا لی۔ تمام کھلاڑیوں نے بہترین فارم دکھائی۔",
    "لاہور میں ہونے والی بارش نے شہر کے کئی علاقوں میں سیلاب کی صورتحال پیدا کر دی۔ نکاسی آب کا نظام ناکام ہونے سے شہریوں کو شدید مشکلات کا سامنا ہے۔ انتظامیہ نے ہنگامی اقدامات شروع کر دیے۔",
    "موسمیاتی تبدیلی کے اثرات پاکستان میں تیزی سے بڑھ رہے ہیں۔ گلیشیئرز پگھلنے سے سیلاب کا خطرہ بڑھ گیا ہے۔ حکومت کو فوری طور پر موثر اقدامات کرنے کی ضرورت ہے تاکہ مستقبل محفوظ بنایا جا سکے۔",
    "ڈیجیٹل پاکستان کے منصوبے کے تحت دور دراز علاقوں میں انٹرنیٹ کی سہولیات فراہم کی جا رہی ہیں۔ اس سے تعلیم اور کاروبار کے شعبوں میں انقلاب آنے کی توقع ہے۔ نوجوانوں کو آن لائن ملازمتوں کے مواقع ملیں گے۔",
    "معیشت اور کاروبار کی خبریں",
    "کرکٹ کی تازہ ترین خبریں",
    "فلموں اور ڈرامے کی خبریں",
    "پاکستان کا انتخابی نظام",
    "پاکستانی شوبز انڈسٹری",
    "صحت اور تندرستی کے حوالے سے مفید معلومات",
    "پاکستان میں تعلیمی نظام اور جدید تربیت",
    "ٹیکنالوجی",
    "کاروبار اور معاشی ترقی کی خبریں",
    "پاکستانی سیاست اور حکومتی پالیسیاں",
    "کھیلوں اور تفریحی پروگراموں کی خبریں",
    "مذہبی تعلیمات اور روحانی معلومات",
    "پاکستان کے خوبصورت سیاحتی مقامات"
    ]


    for idx, query in enumerate(queries, 1):
        print(f"\n{'─'*70}")
        print(f"QUERY {idx}: {query}")
        print(f"{'─'*70}")

        recommender.add_heading(f"Query {idx}: {query}", level=2)

        results = recommender.get_recommendations(
            query=query,
            n_results=10  # Get top 10 for each query
        )

        # Display only headlines for compact view
        if results['ids'] and len(results['ids'][0]) > 0:
            headlines_data = []
            for i, (doc_id, distance, metadata) in enumerate(zip(
                results['ids'][0],
                results['distances'][0],
                results['metadatas'][0]
            ), 1):
                similarity = (1 - distance) * 100
                print(f"{i}. [{similarity:.1f}%] {metadata.get('headline', 'N/A')}")
                headlines_data.append([f"{i}", f"{similarity:.1f}%", metadata.get('headline', 'N/A')])

            # Add headlines table to document
            recommender.add_table(headlines_data, headers=["Rank", "Similarity", "Headline"])
        print()

    # Save the final document
    recommender.save_document()

    print("="*70)
    print("✓ RECOMMENDATION SYSTEM DEMO COMPLETED!")
    print("✓ Using MAX POOLING embeddings for recommendations")
    print("✓ Connected to chroma_db_max database")
    print(f"✓ Comprehensive report saved to: Urdu_News_Recommendations_Max_Pooling_Report.docx")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM - MAX POOLING EMBEDDINGS
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
Cleaning dataset...
✓ Dataset loaded: 111853 articles

INITIALIZING RECOMMENDATION SYSTEM WITH MAX POOLING
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Connecting to ChromaDB at: chroma_db_max
✓ Connected to collection: urdu_news_embeddings_10k_max
✓ Total articles in database: 111853

SYSTEM STATISTICS
total_articles: 111853
collection_name: urdu_news_embeddings_10k_max
model: urduhack/roberta-urdu-small
device: cuda
embedding_dimension: 768
pooling_method: MAX POOLING


MULTIPLE QUERY RECOMMENDATIONS

──────────────────────────────────────────────────────────────────────
QUERY 1: پاکستان میں شمسی توانائی کے منصوبوں کو فروغ دینے کے لیے حکومت نے نئی پالیسی متعارف کرائی ہے۔ اس پالیسی کے تحت گھریلو صارفین کو سستی قرضوں کی سہولت فراہم کی جائے گی۔
──────────────────────────────────────────────────────────────────────

GENERATING RECOMMENDATIONS USING

# **Testing with 10,000 dataset, CLS pooling, urduhack roberta, without dimensionality reduction and with articles divided into batches for capturing complete article’s semantic meaning**

In [9]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION AND STORAGE IN CHROMADB
# Updated for 10,000 records with headline, category, and content columns
# Using CLS POOLING and chroma_db_cls database
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in ChromaDB for efficient retrieval in recommendation systems.

    Designed for large datasets (10,000+ records) with semantic search on content.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_cls"):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path: Path to store ChromaDB locally
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.chroma_db_path.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB client (persistent storage)
        print(f"Initializing ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(
            path=str(self.chroma_db_path)
        )

        # Create or get collection for storing embeddings
        self.collection = self.client.get_or_create_collection(
            name="urdu_news_embeddings_10k_cls",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity for recommendations
        )

    def cls_pooling(self, model_output, attention_mask):
        """
        Apply CLS pooling to model output to get sentence embeddings.

        This takes the [CLS] token embedding from the model output,
        which is specifically trained to represent the entire sequence.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            CLS token embeddings (batch_size, embedding_dim)
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Get the [CLS] token embedding (first token in the sequence)
        cls_embeddings = token_embeddings[:, 0, :]

        return cls_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for a single text using CLS pooling.

        For long articles (>512 tokens), this method splits the text into
        overlapping chunks, generates embeddings for each chunk, and then
        averages them to create a single embedding that represents the entire article.

        Args:
            text: Input Urdu text (content column)
            max_length: Maximum tokens per chunk (default: 512)
            chunk_overlap: Overlap between chunks to maintain context (default: 50)

        Returns:
            Embedding vector as numpy array (768,)
        """
        # First, tokenize to check if text is longer than max_length
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        # If text fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.cls_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long articles: Split into overlapping chunks
        chunk_size = max_length - 2  # Leave room for [CLS] and [SEP] tokens
        stride = chunk_size - chunk_overlap

        chunk_embeddings = []

        # Process text in chunks
        for i in range(0, len(tokens), stride):
            # Get chunk tokens
            chunk_tokens = tokens[i:i + chunk_size]

            # Stop if chunk is too small (less than 50 tokens)
            if len(chunk_tokens) < 50:
                break

            # Decode tokens back to text
            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)

            # Generate embedding for this chunk
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.cls_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        # Average all chunk embeddings to get final embedding
        final_embedding = np.mean(chunk_embeddings, axis=0)

        return final_embedding

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles in dataset and store in ChromaDB.

        Semantic search is performed on the content column, while headline
        and category are stored as metadata for display.

        Args:
            df: Dataframe containing articles with headline, category, and content
            content_column: Name of column containing article content (for embeddings)
            headline_column: Name of column containing article headline (metadata)
            category_column: Name of column containing article category (metadata)
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Content column: '{content_column}' (used for semantic search)")
        print(f"Headline column: '{headline_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (stored as metadata)")
        print(f"Pooling method: CLS POOLING")
        print(f"Database: chroma_db_cls")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data for ChromaDB
        ids = []
        embeddings = []
        metadatas = []
        documents = []

        for idx, row in df.iterrows():
            # Show progress every 500 articles (more frequent for 10k dataset)
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            # Get content for semantic search
            content_text = str(row[content_column])

            # Skip empty content
            if len(content_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty content")
                continue

            try:
                # Generate embedding from content
                embedding = self.generate_embedding_for_text(content_text)

                # Prepare data for ChromaDB
                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings.append(embedding.tolist())

                # Store content preview (first 500 chars)
                documents.append(content_text[:500])

                # Store metadata (headline, category, and article index)
                metadata = {
                    "article_index": idx,
                    "headline": str(row.get(headline_column, "Unknown")),
                    "category": str(row.get(category_column, "Unknown")),
                    "content_length": len(content_text),
                    "pooling_method": "cls_pooling"
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Add all embeddings to ChromaDB in batches (ChromaDB has max batch size limit)
        print(f"\n{'='*70}")
        print(f"STORING {len(ids)} EMBEDDINGS IN CHROMADB...")
        print(f"{'='*70}")

        # ChromaDB has a max batch size limit (~5000), so we add in batches
        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            self.collection.add(
                ids=ids[batch_idx:batch_end],
                embeddings=embeddings[batch_idx:batch_end],
                documents=documents[batch_idx:batch_end],
                metadatas=metadatas[batch_idx:batch_end]
            )

        print(f"✓ All batches stored successfully!")

        total_time = time.time() - start_time
        print(f"\n✓ Successfully stored {len(ids)} embeddings in ChromaDB")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5) -> dict:
        """
        Search for similar articles using query text.

        Searches based on content embeddings and returns results with
        headline, category, and full content.

        Args:
            query_text: Query text to find similar articles
            n_results: Number of similar articles to return

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Search in ChromaDB
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about the ChromaDB collection.

        Returns:
            Dictionary with collection information
        """
        count = self.collection.count()
        return {
            "total_embeddings": count,
            "collection_name": self.collection.name,
            "pooling_method": "CLS POOLING",
            "embedding_dimension": 768,
            "search_column": "content",
            "db_path": str(self.chroma_db_path)
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Load your preprocessed dataset
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM")
    print("USING CLS POOLING AND chroma_db_cls DATABASE")
    print("="*70)
    print("\nLoading balanced dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline'][:100]}...")
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_cls"
    )

    # Generate embeddings and store in ChromaDB
    embedder.generate_embeddings_for_dataset(
        df=df,
        content_column="content",      # Semantic search on content
        headline_column="Headline",    # Store as metadata (capital H)
        category_column="Category"     # Store as metadata (capital C)
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # Test: Search for similar articles with custom query
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH ON CONTENT")
    print("="*70)

    # Custom query - you can modify this as needed
    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"
    print(f"\nQuery: {query}\n")

    results = embedder.search_similar_articles(
        query_text=query,
        n_results=5
    )

    print("Top 5 most relevant articles found:")
    print("="*70)

    if results['ids'] and len(results['ids']) > 0:
        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            print(f"\n{'='*70}")
            print(f"RESULT #{i+1}")
            print(f"{'='*70}")
            print(f"Article ID: {doc_id}")
            print(f"Similarity Score: {1 - distance:.4f} (higher is better)")
            print(f"\nHEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"Pooling Method: {metadata.get('pooling_method', 'cls_pooling')}")

            # Get full content from dataframe
            article_idx = metadata.get('article_index', None)
            if article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                # Display article preview (first 300 characters)
                print(f"\n--- ARTICLE PREVIEW (First 300 chars) ---")
                print(full_content[:300] + "..." if len(full_content) > 300 else full_content)

                # Optionally display full content (commented out by default to avoid clutter)
                # Uncomment the lines below if you want to see full content
                # print(f"\n--- FULL CONTENT ---")
                # print(full_content)
            else:
                print(f"\n--- ARTICLE PREVIEW ---")
                print(document)

            print(f"\n{'='*70}")
    else:
        print("No results found!")

    print("\n" + "="*70)
    print("✓ EMBEDDINGS GENERATION AND STORAGE COMPLETED SUCCESSFULLY!")
    print("✓ Using CLS POOLING for embeddings")
    print("✓ Database stored as: chroma_db_cls")
    print("✓ Semantic search is now available on the content column")
    print("✓ Headlines and categories are stored as metadata")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM
USING CLS POOLING AND chroma_db_cls DATABASE

Loading balanced dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا...
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small


2026-01-16 23:19:39,151 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Initializing ChromaDB at: chroma_db_cls

GENERATING EMBEDDINGS FOR 111853 ARTICLES
Content column: 'content' (used for semantic search)
Headline column: 'Headline' (stored as metadata)
Category column: 'Category' (stored as metadata)
Pooling method: CLS POOLING
Database: chroma_db_cls

Processed 500/111853 articles (5.03s elapsed, ETA: 1121.25s)
Processed 1000/111853 articles (8.75s elapsed, ETA: 969.56s)
Processed 1500/111853 articles (12.22s elapsed, ETA: 898.76s)
Processed 2000/111853 articles (15.38s elapsed, ETA: 844.85s)
Processed 2500/111853 articles (18.36s elapsed, ETA: 803.20s)
Processed 3000/111853 articles (21.36s elapsed, ETA: 774.95s)
Processed 3500/111853 articles (24.23s elapsed, ETA: 750.09s)
Processed 4000/111853 articles (28.23s elapsed, ETA: 761.16s)
Processed 4500/111853 articles (32.47s elapsed, ETA: 774.65s)
Processed 5000/111853 articles (36.49s elapsed, ETA: 779.81s)
Processed 5500/111853 articles (40.10s elapsed, ETA: 775.34s)
Processed 6000/111853 articles (4

# **Recommender System for CLS Pooling**

In [3]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM
# Uses pre-stored ChromaDB embeddings with CLS POOLING to generate recommendations
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime

class UrduNewsRecommender:
    """
    Recommendation system for Urdu news articles using pre-stored CLS POOLING embeddings.
    Connects to existing ChromaDB and generates recommendations based on queries.
    Outputs results to Word document.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_cls",
                 collection_name: str = "urdu_news_embeddings_10k_cls",
                 output_doc_path: str = "Urdu_News_Recommendations_CLS_Pooling_Report.docx"):
        """
        Initialize the recommender with pre-stored CLS POOLING embeddings.

        Args:
            model_name: HuggingFace model identifier (same as used for embedding generation)
            chroma_db_path: Path to existing ChromaDB with cls pooling embeddings
            collection_name: Name of the collection with cls pooling embeddings
            output_doc_path: Path for the output Word document
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer (for query embeddings)
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Connect to existing ChromaDB with CLS POOLING embeddings
        print(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.add_paragraph(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(path=str(self.chroma_db_path))

        # Get existing collection with CLS POOLING embeddings
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Connected to collection: {collection_name}")
            print(f"✓ Total articles in database: {self.collection.count()}")
            self.add_paragraph(f"✓ Connected to collection: {collection_name}")
            self.add_paragraph(f"✓ Total articles in database: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name}'")
            print(f"Make sure embeddings are generated first!")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System Report (CLS Pooling)', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'

        for row_idx, row_data in enumerate(data):
            if headers and row_idx == 0:
                continue  # Skip first row if headers were added
            if not headers:
                cells = table.rows[row_idx].cells
            else:
                cells = table.add_row().cells
            for col_idx, cell_data in enumerate(row_data):
                cells[col_idx].text = str(cell_data)

        return table

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")

    def cls_pooling(self, model_output, attention_mask):
        """
        Apply CLS POOLING to get sentence embeddings.

        This extracts the [CLS] token embedding which is specifically trained
        to represent the entire sequence.
        """
        token_embeddings = model_output[0]
        # Get the [CLS] token embedding (first token in the sequence)
        cls_embeddings = token_embeddings[:, 0, :]
        return cls_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for query text using CLS POOLING.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks

        Returns:
            Query embedding vector using CLS POOLING
        """
        # Tokenize to check length
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # If query fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.cls_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long queries: use chunking (same as article processing)
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.cls_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def get_recommendations(self, query: str, n_results: int = 5,
                          filter_category: str = None) -> dict:
        """
        Get article recommendations based on Urdu query using CLS POOLING embeddings.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            filter_category: Optional category filter (e.g., "Sports", "Business")

        Returns:
            Dictionary containing recommended articles with metadata
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS USING CLS POOLING")
        print(f"{'='*70}")
        print(f"Query: {query}")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Generate query embedding using CLS POOLING
        print("Generating query embedding using CLS POOLING...")
        query_embedding = self.generate_query_embedding(query)
        print("✓ Query embedding generated")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Search in ChromaDB with CLS POOLING embeddings
        print(f"Searching for similar articles...")
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"✓ Found {len(results['ids'][0]) if results['ids'] else 0} recommendations\n")

        return results

    def display_recommendations(self, results: dict, df: pd.DataFrame = None,
                               show_full_content: bool = False):
        """
        Display recommendations in a formatted way.

        Args:
            results: Results from get_recommendations()
            df: Optional dataframe to fetch full content
            show_full_content: Whether to display full article content
        """
        if not results['ids'] or len(results['ids'][0]) == 0:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        print(f"{'='*70}")
        print(f"TOP {len(results['ids'][0])} RECOMMENDATIONS")
        print(f"{'='*70}\n")

        self.add_heading(f"Top {len(results['ids'][0])} Recommendations", level=2)

        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            # Calculate similarity score (1 - distance for cosine)
            similarity_score = 1 - distance

            print(f"{'='*70}")
            print(f"RECOMMENDATION #{i+1}")
            print(f"{'='*70}")
            print(f"📰 Article ID: {doc_id}")
            print(f"🎯 Similarity Score: {similarity_score:.4f} ({similarity_score*100:.2f}%)")
            print(f"\n📌 HEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"📂 CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"📏 Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"🔧 Pooling Method: {metadata.get('pooling_method', 'cls_pooling')}")

            # Add to Word document
            self.add_heading(f"Recommendation #{i+1} (Similarity: {similarity_score*100:.2f}%)", level=3)

            recommendation_data = [
                ["Article ID", doc_id],
                ["Similarity Score", f"{similarity_score:.4f} ({similarity_score*100:.2f}%)"],
                ["Headline", metadata.get('headline', 'N/A')],
                ["Category", metadata.get('category', 'N/A')],
                ["Content Length", f"{metadata.get('content_length', 'N/A')} characters"],
                ["Pooling Method", metadata.get('pooling_method', 'cls_pooling')]
            ]

            self.add_table(recommendation_data)

            # Get and display article content
            article_idx = metadata.get('article_index', None)

            if df is not None and article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                if show_full_content:
                    print(f"\n📄 FULL CONTENT:")
                    print(f"{'-'*70}")
                    print(full_content)
                    self.add_paragraph("Full Content:")
                    self.add_paragraph(full_content)
                else:
                    # Display preview (first 400 characters)
                    preview = full_content[:400] + "..." if len(full_content) > 400 else full_content
                    print(f"\n📄 CONTENT PREVIEW:")
                    print(f"{'-'*70}")
                    print(preview)
                    self.add_paragraph("Content Preview:")
                    self.add_paragraph(preview)
            else:
                # Fallback to stored document preview
                print(f"\n📄 CONTENT PREVIEW:")
                print(f"{'-'*70}")
                print(document)
                self.add_paragraph("Content Preview:")
                self.add_paragraph(document)

            print(f"\n{'='*70}\n")
            self.add_paragraph('')  # Add empty line between recommendations

    def get_statistics(self) -> dict:
        """Get statistics about the recommendation system."""
        return {
            "total_articles": self.collection.count(),
            "collection_name": self.collection.name,
            "model": self.model_name,
            "device": str(self.device),
            "embedding_dimension": 768,
            "pooling_method": "CLS POOLING"
        }

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()
        stats_data = [[key, value] for key, value in stats.items()]
        self.add_table(stats_data, headers=["Statistic", "Value"])


# =============================================================================
# MAIN EXECUTION - RECOMMENDATION SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM - CLS POOLING EMBEDDINGS")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)

    # Load the dataset (optional - only needed for displaying full content)
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    
    # Clean the data - handle NaN values in key columns
    print("Cleaning dataset...")
    df['Category'] = df['Category'].fillna('Unknown')
    df['Headline'] = df['Headline'].fillna('Unknown')
    df['content'] = df['content'].fillna('')
    
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender (connects to existing ChromaDB with CLS POOLING embeddings)
    print("\n" + "="*70)
    print("INITIALIZING RECOMMENDATION SYSTEM WITH CLS POOLING")
    print("="*70)

    recommender = UrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_cls",
        collection_name="urdu_news_embeddings_10k_cls",
        output_doc_path="Urdu_News_Recommendations_CLS_Pooling_Report.docx"
    )

    # Add dataset information to document with NaN handling
    recommender.add_heading("Dataset Information", level=1)
    
    # Get unique categories, handle NaN values
    categories = df['Category'].unique().tolist()
    # Convert all items to string, handle NaN
    categories_str = [str(cat) if pd.notna(cat) else "Unknown" for cat in categories]
    
    dataset_info = [
        ["Total articles", len(df)],
        ["Dataset shape", f"{df.shape}"],
        ["Unique categories", df['Category'].nunique()],
        ["Categories", ", ".join(categories_str[:15]) + ("..." if len(categories_str) > 15 else "")]
    ]
    recommender.add_table(dataset_info, headers=["Metric", "Value"])

    # Display system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    for key, value in stats.items():
        print(f"{key}: {value}")

    recommender.add_statistics_to_doc()

    # =================================================================
    # EXAMPLE 3: Multiple queries (batch recommendations)
    # =================================================================
    print("\n\n" + "="*70)
    print("MULTIPLE QUERY RECOMMENDATIONS")
    print("="*70)

    recommender.add_heading("Example 3: Multiple Query Recommendations", level=1)

    queries = [
    "پاکستان میں شمسی توانائی کے منصوبوں کو فروغ دینے کے لیے حکومت نے نئی پالیسی متعارف کرائی ہے۔ اس پالیسی کے تحت گھریلو صارفین کو سستی قرضوں کی سہولت فراہم کی جائے گی۔",
    "کرکٹ ورلڈ کپ میں پاکستان کی ٹیم نے شاندار کارکردگی کا مظاہرہ کیا۔ بابر اعظم کی کپتانی میں ٹیم نے مسلسل پانچ میچ جیت کر سیمی فائنل میں جگہ بنا لی۔ تمام کھلاڑیوں نے بہترین فارم دکھائی۔",
    "لاہور میں ہونے والی بارش نے شہر کے کئی علاقوں میں سیلاب کی صورتحال پیدا کر دی۔ نکاسی آب کا نظام ناکام ہونے سے شہریوں کو شدید مشکلات کا سامنا ہے۔ انتظامیہ نے ہنگامی اقدامات شروع کر دیے۔",
    "موسمیاتی تبدیلی کے اثرات پاکستان میں تیزی سے بڑھ رہے ہیں۔ گلیشیئرز پگھلنے سے سیلاب کا خطرہ بڑھ گیا ہے۔ حکومت کو فوری طور پر موثر اقدامات کرنے کی ضرورت ہے تاکہ مستقبل محفوظ بنایا جا سکے۔",
    "ڈیجیٹل پاکستان کے منصوبے کے تحت دور دراز علاقوں میں انٹرنیٹ کی سہولیات فراہم کی جا رہی ہیں۔ اس سے تعلیم اور کاروبار کے شعبوں میں انقلاب آنے کی توقع ہے۔ نوجوانوں کو آن لائن ملازمتوں کے مواقع ملیں گے۔",
    "معیشت اور کاروبار کی خبریں",
    "کرکٹ کی تازہ ترین خبریں",
    "فلموں اور ڈرامے کی خبریں",
    "پاکستان کا انتخابی نظام",
    "پاکستانی شوبز انڈسٹری",
    "صحت اور تندرستی کے حوالے سے مفید معلومات",
    "پاکستان میں تعلیمی نظام اور جدید تربیت",
    "ٹیکنالوجی",
    "کاروبار اور معاشی ترقی کی خبریں",
    "پاکستانی سیاست اور حکومتی پالیسیاں",
    "کھیلوں اور تفریحی پروگراموں کی خبریں",
    "مذہبی تعلیمات اور روحانی معلومات",
    "پاکستان کے خوبصورت سیاحتی مقامات"
    ]

    for idx, query in enumerate(queries, 1):
        print(f"\n{'─'*70}")
        print(f"QUERY {idx}: {query}")
        print(f"{'─'*70}")

        recommender.add_heading(f"Query {idx}: {query}", level=2)

        results = recommender.get_recommendations(
            query=query,
            n_results=10  # Get top 10 for each query
        )

        # Display only headlines for compact view
        if results['ids'] and len(results['ids'][0]) > 0:
            headlines_data = []
            for i, (doc_id, distance, metadata) in enumerate(zip(
                results['ids'][0],
                results['distances'][0],
                results['metadatas'][0]
            ), 1):
                similarity = (1 - distance) * 100
                print(f"{i}. [{similarity:.1f}%] {metadata.get('headline', 'N/A')}")
                headlines_data.append([f"{i}", f"{similarity:.1f}%", metadata.get('headline', 'N/A')])

            # Add headlines table to document
            recommender.add_table(headlines_data, headers=["Rank", "Similarity", "Headline"])
        print()

    # Save the final document
    recommender.save_document()

    print("="*70)
    print("✓ RECOMMENDATION SYSTEM DEMO COMPLETED!")
    print("✓ Using CLS POOLING embeddings for recommendations")
    print("✓ Connected to chroma_db_cls database")
    print(f"✓ Comprehensive report saved to: Urdu_News_Recommendations_CLS_Pooling_Report.docx")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM - CLS POOLING EMBEDDINGS
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
Cleaning dataset...
✓ Dataset loaded: 111853 articles

INITIALIZING RECOMMENDATION SYSTEM WITH CLS POOLING
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Connecting to ChromaDB at: chroma_db_cls
✓ Connected to collection: urdu_news_embeddings_10k_cls
✓ Total articles in database: 111853

SYSTEM STATISTICS
total_articles: 111853
collection_name: urdu_news_embeddings_10k_cls
model: urduhack/roberta-urdu-small
device: cuda
embedding_dimension: 768
pooling_method: CLS POOLING


MULTIPLE QUERY RECOMMENDATIONS

──────────────────────────────────────────────────────────────────────
QUERY 1: پاکستان میں شمسی توانائی کے منصوبوں کو فروغ دینے کے لیے حکومت نے نئی پالیسی متعارف کرائی ہے۔ اس پالیسی کے تحت گھریلو صارفین کو سستی قرضوں کی سہولت فراہم کی جائے گی۔
──────────────────────────────────────────────────────────────────────

GENERATING RECOMMENDATIONS USING